# 🦜🔗 LangChain 101: Comprehensive Guide with Local Ollama

Welcome to **LangChain 101**! This notebook provides a complete, hands-on tutorial covering all core concepts of **LangChain** using **local Ollama models** (`nemotron-3-nano:4b`, `gemma4:e2b`, and `nomic-embed-text:latest`).

### 🎯 Learning Objectives:
1. **Model Connection & Basics**: Invoking, streaming, and batching with `ChatOllama`.
2. **Prompts & Messages**: `ChatPromptTemplate`, message types, and `MessagesPlaceholder`.
3. **Output Parsers**: `StrOutputParser` and Pydantic-based `Structured Output`.
4. **LCEL (LangChain Expression Language)**: The pipe operator (`|`), `RunnableParallel`, `RunnablePassthrough`, and `RunnableLambda`.
5. **Memory & Chat History**: Conversation management with `InMemoryChatMessageHistory` & `RunnableWithMessageHistory`.
6. **Tool Calling & Custom Tools**: Creating `@tool` functions and binding them to LLMs.
7. **Agent Construction**: Building interactive, tool-calling agents.
8. **RAG (Retrieval-Augmented Generation)**: Splitting documents, vector embeddings with `OllamaEmbeddings`, vector stores, and retriever chains.


--- 
## 0. Setup & Local Ollama Verification

First, let's set up imports and initialize our local Ollama model instance. We can use `nemotron-3-nano:4b` or `gemma4:e2b`.

In [ ]:
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Select local Ollama model
MODEL_NAME = "nemotron-3-nano:4b"  # Or "gemma4:e2b"
EMBED_MODEL = "nomic-embed-text:latest"

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0.7
)

print(f"✅ Connected to local Ollama model: {MODEL_NAME}")


--- 
## 1. Model Invocation: `invoke`, `stream`, & `batch` 

LangChain models implement the standard **Runnable interface**, giving you uniform methods to interact with models:

In [ ]:
# 1. Direct Invoke
response = llm.invoke("Give me a 1-sentence definition of Artificial Intelligence.")
print("--- Standard Response ---")
print(response.content)

# 2. Stream Response
print("\n--- Streaming Response ---")
for chunk in llm.stream("Count from 1 to 5 rapidly."):
    print(chunk.content, end="", flush=True)
print()

# 3. Batch Invocations
print("\n--- Batch Responses ---")
questions = [
    "What is Python?",
    "What is Ollama?"
]
batch_results = llm.batch(questions)
for q, res in zip(questions, batch_results):
    print(f"Q: {q}\nA: {res.content.strip()}\n")


--- 
## 2. Prompts & Messages: `ChatPromptTemplate`

In modern AI applications, structured chat messages (`SystemMessage`, `HumanMessage`, `AIMessage`) guide LLM behavior.
`ChatPromptTemplate` parameterizes system instructions and user inputs.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Creating a ChatPromptTemplate with System & Human roles
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful programming tutor skilled in {language}. Keep explanations concise."),
    ("human", "Explain {concept} in 2 simple bullet points.")
])

# Formatting the prompt
formatted_messages = prompt_template.format_messages(language="Python", concept="Decorators")
print("Formatted Prompt Messages:", formatted_messages)

# Invoking the model with the formatted prompt
response = llm.invoke(formatted_messages)
print("\nModel Response:")
print(response.content)


--- 
## 3. Output Parsers & Pydantic Structured Output

Raw LLM outputs are `AIMessage` objects. Output parsers convert LLM outputs into clean strings, dictionaries, or typed Pydantic models.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

# 1. StrOutputParser: Extracts string content from AIMessage
str_parser = StrOutputParser()
clean_text = str(str_parser.invoke(response))
print("Parsed String Content:", type(clean_text), f"'{clean_text[:50]}...'")

# 2. Structured Output with Pydantic
class ProgrammingConcept(BaseModel):
    name: str = Field(description="The name of the concept")
    language: str = Field(description="Programming language")
    summary: str = Field(description="1-sentence summary of the concept")
    difficulty: str = Field(description="Beginner, Intermediate, or Advanced")

# Bind Pydantic schema to model
structured_llm = llm.with_structured_output(ProgrammingConcept)

result = structured_llm.invoke("Explain Recursion in Python")
print("\nStructured Pydantic Output:")
print(f"Concept: {result.name} ({result.language})")
print(f"Difficulty: {result.difficulty}")
print(f"Summary: {result.summary}")


--- 
## 4. LangChain Expression Language (LCEL)

LCEL allows you to compose complex chains using the pipe operator `|`.
A standard chain follows: `Prompt | Model | OutputParser`.

You can also use:
- `RunnableParallel`: Run multiple steps in parallel.
- `RunnablePassthrough`: Pass inputs through unchanged.
- `RunnableLambda`: Wrap custom Python functions as runnable components.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

# 1. Basic LCEL Chain
chain1 = prompt_template | llm | StrOutputParser()
output1 = chain1.invoke({"language": "Python", "concept": "List Comprehensions"})
print("--- LCEL Chain 1 Output ---")
print(output1)

# 2. Advanced LCEL: Parallel Composition
poem_prompt = ChatPromptTemplate.from_template("Write a 2-line poem about {topic}.")
joke_prompt = ChatPromptTemplate.from_template("Tell a 1-line joke about {topic}.")

poem_chain = poem_prompt | llm | StrOutputParser()
joke_chain = joke_prompt | llm | StrOutputParser()

# Combine in parallel
combined_chain = RunnableParallel(
    poem=poem_chain,
    joke=joke_chain
)

parallel_output = combined_chain.invoke({"topic": "Artificial Intelligence"})
print("\n--- Parallel Output ---")
print("POEM:\n", parallel_output['poem'])
print("JOKE:\n", parallel_output['joke'])


--- 
## 5. Memory & Conversation History

LLMs are stateless by default. To maintain conversation context across user turns, we use `InMemoryChatMessageHistory` and wrap our LCEL chain with `RunnableWithMessageHistory`.

In [ ]:
from langchain_community.chat_message_histories import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Session store for chat histories
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Prompt expecting chat history
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly AI companion. Answer questions contextually."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chat_chain = conversational_prompt | llm | StrOutputParser()

# Conversational Chain wrapped with message history
with_history_chain = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Turn 1
res1 = with_history_chain.invoke(
    {"input": "Hi! My name is Alice and I live in San Francisco."},
    config={"configurable": {"session_id": "user_123"}}
)
print("Turn 1 Response:", res1)

# Turn 2 (Testing Context Memory)
res2 = with_history_chain.invoke(
    {"input": "What is my name and where do I live?"},
    config={"configurable": {"session_id": "user_123"}}
)
print("\nTurn 2 Response (Memory Check):")
print(res2)


--- 
## 6. Tools & Tool Calling

Tools let LLMs interact with external APIs, databases, or python logic.
Using `@tool` decorator, we can define custom python functions and bind them to the local model.

In [ ]:
from langchain_core.tools import tool

# Define custom tools
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

tools = [add_numbers, multiply_numbers]

# Bind tools to local LLM
llm_with_tools = llm.bind_tools(tools)

# Invoke model with query that requires tool
tool_response = llm_with_tools.invoke("What is 35 multiplied by 12?")
print("Tool Call Output from Model:")
print(f"Tool Calls: {tool_response.tool_calls}")


--- 
## 7. Building an Interactive Tool-Calling Agent Loop

An **Agent** inspects the user query, decides which tool(s) to call, executes the tool, and uses the tool's return value to synthesize the final response.

In [ ]:
from langchain_core.messages import ToolMessage

def run_simple_agent(query: str):
    print(f"User Query: '{query}'")
    messages = [HumanMessage(content=query)]
    
    # Step 1: LLM decides tool call
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)
    
    if ai_msg.tool_calls:
        for tool_call in ai_msg.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f"🤖 Executing Tool: {tool_name}({tool_args})")
            
            # Match tool name and execute
            if tool_name == "add_numbers":
                result = add_numbers.invoke(tool_args)
            elif tool_name == "multiply_numbers":
                result = multiply_numbers.invoke(tool_args)
            else:
                result = "Tool not found"
                
            # Append ToolMessage with result
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call['id']))
            
        # Step 2: Final response generation
        final_response = llm.invoke(messages)
        print("Final Answer:", final_response.content)
    else:
        print("Direct Answer:", ai_msg.content)

run_simple_agent("What is 150 multiplied by 4?")


--- 
## 8. Retrieval-Augmented Generation (RAG)

RAG provides domain-specific knowledge to your LLM by retrieving context from external documents.

Steps in RAG:
1. **Load & Chunk Text**: Split documents into chunks (`RecursiveCharacterTextSplitter`).
2. **Vector Embeddings**: Compute embeddings with local `OllamaEmbeddings` (`nomic-embed-text:latest`).
3. **Store in Retriever**: Store chunks in a vector store.
4. **LCEL RAG Chain**: Pass retrieved context to the LLM.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

# Sample Document Knowledge Base
raw_text = """
The GenAI Project Architecture Guide:
1. Local Model Engine: Powered by Ollama running gemma4:e2b and nemotron-3-nano:4b.
2. Vector Search Engine: Utilizes nomic-embed-text for fast 768-dimensional local text embeddings.
3. Agent Framework: Uses LangChain & LangGraph for orchestration, supporting tools and multi-turn memory.
4. Security Protocol: All data remains local with zero cloud API dependency.
"""

# 1. Split Text into Chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
docs = text_splitter.create_documents([raw_text])
print(f"Split document into {len(docs)} chunks.")

# 2. Local Vector Embeddings & Vector Store
embeddings = OllamaEmbeddings(model=EMBED_MODEL)
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Construct LCEL RAG Chain
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question strictly based on the following context:

Context:
{context}

Question: {question}
Answer:
""")

def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Query RAG Chain
rag_result = rag_chain.invoke("What is the vector search engine and embedding dimension used in the project?")
print("\n--- RAG Answer ---")
print(rag_result)


--- 
## 9. Summary & Next Steps

🎉 **Congratulations!** You have covered all foundational concepts of **LangChain**:
- Model invocation (`invoke`, `stream`, `batch`)
- `ChatPromptTemplate` and message structures
- Output parsing & Pydantic structured output
- LCEL pipelines (`|`, `RunnableParallel`, `RunnablePassthrough`)
- Memory management with `RunnableWithMessageHistory`
- `@tool` integration and agent loops
- Complete local RAG pipeline with `OllamaEmbeddings` and `InMemoryVectorStore`

### ➡️ Next Tutorial Steps:
- **LangGraph 101**: Stateful, cyclic graph agents and human-in-the-loop workflows.
- **AutoGen 101**: Multi-agent conversational patterns with local Ollama models.